# Neoclassical transport and bootstrap current with NEO

How much bootstrap current does a VEST discharge actually carry, and can the
analytic formulas people reach for be trusted at VEST's aspect ratio?

This notebook answers the first from one kinetic state, by running two things
over it and comparing them: the **Sauter and Redl analytic models**, and **NEO**,
a drift-kinetic solver from the GACODE suite.

The second question it answers differently from how it was posed. Which model
looks closer to NEO turns out to be decided by a modelling assumption the state
does not carry -- the impurity content behind `Z_eff` -- rather than by the
models. Section 5 measures that, and section 7 says what survives it.

Three kinds of number are kept apart throughout, because conflating them is the
easiest way to overstate what a calculation shows:

| | |
| --- | --- |
| **Measured** | electron density and temperature, ion temperature -- fitted from Thomson and CX |
| **Modelling assumption** | the effective charge, the impurity that carries it, and where to truncate |
| **Computed** | the bootstrap current and the conductivity, from each model |

Reading order is top to bottom. Sections 1-3 build the input and run the solver;
sections 4-6 compare the models; section 7 says what the comparison does and does
not establish.

## 0. Setup

NEO is an external Fortran executable. VAFT resolves `$GACODEHOME/neo/bin/neo`,
the same `$XHOME` convention CHEASE, EFIT, GPEC, TES and NUBEAM follow -- with
one difference worth knowing: **GACODE builds in place**, so `$GACODEHOME` is the
source checkout itself, and each suite member carries its own `bin`. It also
needs `$GACODE_PLATFORM`, the tag the tree was built with.

`install/gacode/macos.sh` builds it and `install/check_gacode.py` verifies the
result. Without an installation the notebook explains what each section would
show and stops -- every later cell is guarded.

The kinetic state is packaged: `vaft/data/kineticEfit/ods_48224_300ms.json`,
shot 48224 at 300 ms. It is a **repository-only** asset, not shipped in the
wheel.

In [ ]:
import contextlib
import os
import tempfile
from pathlib import Path

try:
    _ipython = get_ipython()
except NameError:
    _ipython = None
if _ipython is None:
    os.environ.setdefault("MPLBACKEND", "Agg")
elif "IPKernelApp" in _ipython.config:
    _ipython.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
import numpy as np
from omas import load_omas_json

import vaft
import vaft.omas
from vaft.code.gacode import find_gacode_executable, neo
from vaft.code.gacode.inputs import prepare_gacode_profile

plt.rcParams["figure.dpi"] = 110

# Modelling assumptions, named here rather than buried in a call below.
#
# Z_EFF: the packaged state carries no zeff profile, and the VEST policy in
#   vest.yaml assumes 1 % carbon and 1 % oxygen -- both marked `assumed`. 2.0 is
#   the value the rest of VAFT's ohmic estimates use. It is an input, not a
#   measurement, and every number below inherits it.
# IMPURITY: how that Z_eff is realised, and the part that bites (issue #803).
#   NEO builds its collision operator from the *species list*; it never reads the
#   z_eff column of input.gacode. Asking for Z_eff = 2 without naming an impurity
#   writes 2 into a column the solver ignores and hands it pure hydrogen at
#   Z_eff = 1. Naming carbon instead gives the main ion and the impurity the
#   densities that satisfy quasi-neutrality and Z_eff together -- n_H = 0.8 n_e
#   and n_C = n_e/30 -- so the plasma NEO solves is the plasma we asked for.
# RHO_MAX: the fitted profiles reach exactly zero at the boundary, and GACODE
#   takes logarithmic gradients, so the grid has to stop short of it. The
#   converter refuses rather than clipping silently; this is that decision made
#   explicitly.
Z_EFF = 2.0
IMPURITY = "C"
RHO_MAX = 0.95

# The surfaces NEO solves. One surface is the default and is cheap; a profile
# needs several, spaced linearly between the two radii.
N_RADIAL, RMIN_1, RMIN_2 = 7, 0.2, 0.8

SAMPLE = vaft.data.data_path("kineticEfit/ods_48224_300ms.json")
CONFIG = neo.NEOConfig(
    n_radial=N_RADIAL, rmin_over_a=RMIN_1, rmin_over_a_2=RMIN_2,
    platform=os.environ.get("GACODE_PLATFORM"),
)

# find_gacode_executable returns None when nothing is configured, and raises when
# $GACODEHOME is set but the tree is not built -- an unbuilt checkout is a
# different problem from an absent one, and says so.
try:
    EXECUTABLE = find_gacode_executable(CONFIG, "neo")
except (FileNotFoundError, PermissionError) as error:
    print(f"GACODE is configured but not usable:\n  {error}\n")
    EXECUTABLE = None
HAVE_GACODE = EXECUTABLE is not None

print(f"sample      {SAMPLE.name}  ({'present' if SAMPLE.exists() else 'MISSING'})")
print(f"surfaces    {N_RADIAL} between r/a {RMIN_1} and {RMIN_2}")
print(f"assumptions Z_eff = {Z_EFF} as a {IMPURITY} impurity, rho_max = {RHO_MAX}")
print(f"executable  {EXECUTABLE}")
if not HAVE_GACODE:
    print(
        "\nNEO was not found. Set $GACODEHOME to a built GACODE checkout and "
        "$GACODE_PLATFORM\nto the tag it was built with -- install/gacode/README.md "
        "explains both -- and\nre-run. Every section below reports what it would show "
        "and skips."
    )

## 1. The kinetic state, and what is wrong with it

Shot 48224 is VAFT's canonical kinetic reference: a magnetic reconstruction, a
kinetic EFIT and a CHEASE refinement at 0.300 s, with seven Thomson channels and
forty charge-exchange channels behind the fitted profiles.

Four caveats travel with it, and they bound everything below:

- **The innermost surfaces are not trustworthy.** `psi_N < 0.05` carries an
  unphysical near-axis `dvolume_dpsi` ramp traceable to a `q(0)` outlier
  (issue #317). Every comparison here excludes it.
- **There are no magnetics.** No pre-EFIT product is packaged, so the
  reconstruction's own input quality cannot be assessed offline and there is no
  diagnostic-fit arm to this study.
- **It is a frozen artifact.** Produced by an earlier OMFIT-backed recipe, it
  carries derived leaves the current native recipe does not reproduce
  byte-for-byte, and a few the installed data dictionary no longer recognises --
  which is why it loads with `consistency_check=False`.
- **It is a cold discharge.** `T_e` is about 65 eV on axis and under 10 eV at
  mid-radius. Expect a small bootstrap current; that is the physics, not a bug.

In [ ]:
ods = load_omas_json(str(SAMPLE), consistency_check=False)

eq = ods["equilibrium.time_slice.0.profiles_1d"]
cp = ods["core_profiles.profiles_1d.0"]
rho = np.asarray(eq["rho_tor_norm"])
te = np.asarray(cp["electrons.temperature"])
ne = np.asarray(cp["electrons.density_thermal"])
f_trap = np.asarray(eq["trapped_fraction"])
ip = float(ods["equilibrium.time_slice.0.global_quantities.ip"])
TIME = float(np.ravel(ods["equilibrium.time"])[0])

print(f"time            {TIME:.3f} s")
print(f"plasma current  {ip / 1e3:.1f} kA")
print(f"T_e             {te[0]:.0f} eV on axis, {np.interp(0.5, rho, te):.0f} eV at rho = 0.5")
print(f"n_e             {ne[0]:.2e} m^-3 on axis")
print(f"trapped fraction {f_trap[1]:.2f} near axis -> {f_trap[-1]:.2f} at the edge")
print()
print("The trapped fraction is what makes this interesting: it reaches 0.9, far")
print("outside the range the 1999 Sauter fit was built on.")

## 2. Convert to the GACODE profile format

`input.gacode` is the GACODE suite's shared profile format -- NEO, TGLF and CGYRO
all read it. It is an interoperability format, not VAFT's kinetic state: the
canonical state stays in IMAS, and the conversion is a deterministic projection
of it.

Three things the converter refuses to do quietly, each of which would otherwise
produce a plausible-looking wrong answer:

- **It will not write `sqrt(psi_N)` under the name `rho`.** GACODE's radial
  coordinate is `sqrt(Phi/Phi_boundary)`; several packaged VAFT equilibria store
  a `sqrt(psi_N)` proxy under `rho_tor_norm`, and the two agree only for a
  flat-`q` cylinder (issues #276, #420).
- **It will not pair slices that are not simultaneous.** The requested time, the
  equilibrium slice and the `core_profiles` slice are resolved separately and
  all three are recorded.
- **It will not clip a zero density or temperature.** GACODE takes logarithmic
  gradients, so it refuses and names the grid point -- which is why `RHO_MAX`
  exists above.

It also converts the sign convention. VAFT holds COCOS 11; `input.gacode` is
COCOS 2, so the toroidal field, the current, `fpol`, the toroidal flux and the
poloidal flux all change sign on the way out. Skipping that would hand NEO a
mirrored device.

In [ ]:
profile = prepare_gacode_profile(ods, rho_max=RHO_MAX, z_eff=Z_EFF, impurity=IMPURITY)
hydrogen_profile = prepare_gacode_profile(ods, rho_max=RHO_MAX, z_eff=Z_EFF)


def realized_charge(p):
    """Z_eff as the species list gives it -- which is the only one NEO reads."""
    n_i, z = np.asarray(p.ni, dtype=float), np.asarray(p.z, dtype=float)
    return float(((n_i * z[:, None] ** 2).sum(axis=0) / np.asarray(p.ne))[0])


print(f"radial points   {profile.n_exp}  (from {rho.size}; rho_max = {RHO_MAX})")
print(f"torfluxa        {profile.torfluxa:+.4g} Wb/rad")
print(f"bcentr          {profile.bcentr:+.4g} T")
print(f"current         {profile.current:+.4g} MA")
print()
print("What the z_eff column says, and what the species list actually gives NEO:")
print(f"  {'species':22s} {'z_eff column':>12s} {'Z_eff NEO sees':>15s}")
for label, p in ((f"with {IMPURITY}", profile), ("without an impurity", hydrogen_profile)):
    print(f"  {', '.join(p.name):22s} {float(np.asarray(p.z_eff)[0]):12.2f}"
          f" {realized_charge(p):15.2f}")
print()
print("The second row is the one that has been quietly wrong: a file declaring")
print("z_eff = 2 beside a single hydrogenic ion is a Z_eff = 1 plasma to the solver.")
print()
print("Signs, after the COCOS 11 -> 2 transform:")
print(f"  IMAS b0 {float(np.ravel(ods['equilibrium.vacuum_toroidal_field.b0'])[0]):+.4g} T"
      f" and ip {ip / 1e6:+.4g} MA -- both counter-clockwise from above")
print(f"  written as bcentr {profile.bcentr:+.4g} and current {profile.current:+.4g},"
      f" which is how\n  expro reads that same orientation back")
print()
for name in ("z_eff", "ni", "rho_max", "torfluxa"):
    if name in profile.provenance:
        print(f"{name:10s} {profile.provenance[name]}")
what_is_missing = profile.missing()
print(f"\nrecorded as unavailable: {', '.join(what_is_missing) if what_is_missing else 'nothing'}")

## 3. Run NEO

`run_neo_case` stages `input.gacode` and `input.neo` into a working directory and
drives the launcher, `$GACODEHOME/neo/bin/neo`, rather than the Fortran binary
underneath it. The launcher expands `input.neo` into the file the binary actually
reads and stamps `out.neo.version` with the revision, platform and date, which is
where the run's executable identity comes from.

One behaviour worth stating, because it is the opposite of what an exit code
usually means: **NEO reports its own input errors by writing them to
`out.neo.run` and then exiting zero.** The adapter reads that log, and treats a
run as solved only when NEO logged no error, wrote its transport product, and the
current it wrote is finite -- the last because a degenerate geometry produces
NaN with nothing logged at all.

In [ ]:
_cleanup = contextlib.ExitStack()
RUN_DIR = None
result = None
hydrogen_result = None

if HAVE_GACODE:
    RUN_DIR = Path(_cleanup.enter_context(tempfile.TemporaryDirectory(prefix="vaft-neo-")))
    print(f"work directory  {RUN_DIR}")
    # The plasma this notebook is about: hydrogen with carbon at Z_eff = 2.
    result = neo.run_neo_case(profile, RUN_DIR / "carbon", CONFIG, check=False)
    # And the same case without the impurity, as the sensitivity arm of section 5.
    # It is not a worse run -- it is a different plasma, and the difference between
    # them turns out to be larger than the difference between the models.
    hydrogen_result = neo.run_neo_case(
        hydrogen_profile, RUN_DIR / "hydrogen", CONFIG, check=False
    )
    for label, r in (("carbon", result), ("hydrogen", hydrogen_result)):
        if not r.ok:
            print(f"\nNEO did not produce a usable {label} result (rc={r.returncode}).")
            for line in getattr(r.outputs_native, "errors", ()) or ():
                print(f"  {line}")

HAVE_RUN = result is not None and result.ok
HAVE_SENSITIVITY = HAVE_RUN and hydrogen_result is not None and hydrogen_result.ok
native = result.outputs_native if result is not None else None

if HAVE_RUN:
    print(f"revision        {native.version['revision']}")
    print(f"platform        {native.version['platform']}")
    print(f"surfaces        {native.grid.n_radial} at r/a"
          f" {np.round(native.grid.r_over_a, 3)}")
    print(f"trapped fraction {native.trapped_fraction:.3f} at the reference surface")
    print(f"products        {len(native.files)} out.neo.* files")
    print()
    # Read back from the run rather than assumed: this is the number the analytic
    # models must be evaluated at for the comparison to be of one plasma.
    print(f"Z_eff NEO used  {float(np.nanmean(native.effective_charge)):.3f}"
          f"  ({native.n_species} species)")
    if HAVE_SENSITIVITY:
        other = hydrogen_result.outputs_native
        print(f"  and {float(np.nanmean(other.effective_charge)):.3f} in the"
              f" impurity-free arm ({other.n_species} species)")
else:
    print("Would run NEO at the surfaces above and report its revision, the radii"
          " solved,\nthe trapped fraction it integrated from the flux surfaces, and"
          " the Z_eff its\nspecies list implies -- twice, once with carbon and once"
          " without.")

## 4. Map the result into IMAS

NEO's native container is the source of truth, in NEO's own normalisation. The
mapping writes only what has a defensible IMAS home, and reports the rest as
skipped rather than leaving it to be inferred from an absent field.

The bootstrap current is **derived, not copied**, twice over. NEO's normalised
`jparB` dimensionalises -- with GACODE's own constants -- to `<J.B>/B_unit`,
while IMAS defines `core_profiles.j_bootstrap` as `average(J.B)/B0` with `B0` the
*vacuum* field. On VEST those differ by about a factor of two **and a sign**,
since `B_unit` is negative under the GACODE convention and varies across the
profile.

In [ ]:
if HAVE_RUN:
    from vaft.machine_mapping.neoclassical import core_profiles_from_neo, core_transport_from_neo

    # The time written is the slice's own, read back in section 1 rather than
    # repeated as a literal here: a mapped result labelled with a time the ODS
    # does not hold is exactly what a nearest-time lookup then answers wrongly.
    profiles_report = core_profiles_from_neo(ods, result, time=TIME, time_index=0)
    transport_report = core_transport_from_neo(ods, result, time=TIME, time_index=0)

    print("written to core_profiles :", ", ".join(profiles_report["written"]) or "nothing")
    print("written to core_transport:", ", ".join(transport_report["written"]) or "nothing")
    print("\nnot written, and why:")
    for reason in profiles_report["skipped"] + transport_report["skipped"]:
        print(f"  - {reason}")

    b_unit = float(native.normalisation.b_unit[0])
    b0 = float(np.ravel(ods["equilibrium.vacuum_toroidal_field.b0"])[0])
    print(f"\nB_unit {b_unit:+.4f} T against the vacuum B0 {b0:+.4f} T"
          f"  ->  factor {b_unit / b0:+.2f}")
    print("A copy would have been wrong by exactly that, sign included.")
else:
    print("Would map the drift-kinetic bootstrap current into core_profiles and the")
    print("particle and energy fluxes into core_transport, and list what it did not map.")

## 5. Compare the models

This is the question the whole exercise exists to answer. Three profiles now sit
on one radial axis: NEO's drift-kinetic solve, and the Sauter and Redl analytic
models evaluated from the same ODS.

The comparison reports **numbers, not verdicts**. A gap between a fitted model
and a drift-kinetic solve is a property of the models; turning it into a
pass/fail would be asserting that one of them is broken. Tolerances, if a caller
wants them, are supplied separately.

Every metric is normalised by the profile's own scale. The bootstrap current
changes sign near the axis, so a pointwise ratio explodes at the crossing --
about 487 % on this state -- and a log ratio is undefined where the current is
negative.

In [ ]:
from vaft.validation.neoclassical import bootstrap_models, model_comparison

# solver_charge= is not decoration: bootstrap_models refuses to compare when the
# analytic charge and the solver's disagree, because that is two plasmas and the
# result reads as a model difference (#803).
SOLVER_CHARGE = float(np.nanmean(native.effective_charge)) if HAVE_RUN else Z_EFF
# The charge alone is not the plasma: carbon and oxygen reach Z_eff = 2 with
# different ion fractions, and an analytic side with no impurity has n_i = n_e
# whatever its Z_eff says. Both are checked.
SOLVER_ION_FRACTION = float(np.nanmean(native.ion_density_fraction)) if HAVE_RUN else None
models = bootstrap_models(
    ods, z_eff=Z_EFF, impurity=IMPURITY, rho_range=(0.05, 1.0),
    solver_charge=SOLVER_CHARGE if HAVE_RUN else None,
    solver_ion_fraction=SOLVER_ION_FRACTION,
)
print("series:", ", ".join(sorted(models["series"])))

if HAVE_RUN:
    comparison = model_comparison(models, reference="neo")
    print(f"\nAgainst NEO, over the {comparison['overlap']} radii it solved,"
          f" both sides at Z_eff = {SOLVER_CHARGE:.2f}:\n")
    print(f"  {'model':8s} {'integrated':>12s} {'RMS/peak':>10s}")
    for name in ("sauter", "redl"):
        entry = comparison["models"][name]
        if entry["integrated_relative_difference"] is None:
            # Fewer than two radii where both profiles are finite -- a narrower
            # RMIN_1/RMIN_2 band produces exactly this. Say so instead of
            # multiplying None by a hundred.
            print(f"  {name:8s} {entry.get('reason', 'no overlap with NEO')}")
            continue
        print(f"  {name:8s} {entry['integrated_relative_difference'] * 100:+11.2f}%"
              f" {entry['rms_over_peak'] * 100:9.2f}%")

    effect = comparison["effect_size"]
    print(f"\nBootstrap current  {effect['integrated_current'] / 1e3:.2f} kA"
          f"  =  {effect['integrated_current'] / ip * 100:.1f}% of Ip")
    print(f"peaking at rho = {effect['peak_rho']:.2f}")
else:
    comparison = model_comparison(models, reference="sauter")
    redl = comparison["models"]["redl"]
    print("\nWithout NEO, the two analytic models can still be compared:")
    print(f"  Redl vs Sauter: {redl['integrated_relative_difference'] * 100:+.2f}%"
          f" integrated, {redl['rms_over_peak'] * 100:.2f}% RMS/peak")
    print("\nWould additionally compare both against NEO's drift-kinetic solve,"
          " twice.")

### What actually decides which model looks closer

Before reading anything into the ranking above, run the same comparison against
the impurity-free plasma and put the two side by side.

The two arms differ only in one modelling choice -- whether `Z_eff = 2` is
realised as a carbon impurity or left as a number in a column NEO never reads.
Everything else is identical: the same equilibrium, the same fitted profiles, the
same surfaces, the same solver revision.

If the ranking survives that, it says something about the models. If it does not,
it says the impurity assumption dominates, and no ranking from this state can be
quoted without it.

In [ ]:
def rank_against(run, charge, impurity):
    """Sauter and Redl against one NEO run, both sides at that run's own charge."""
    state = load_omas_json(str(SAMPLE), consistency_check=False)
    core_profiles_from_neo(state, run, time=TIME, time_index=0)
    rows = bootstrap_models(
        state, z_eff=charge, impurity=impurity, rho_range=(0.05, 1.0),
        solver_charge=charge,
        solver_ion_fraction=float(np.nanmean(run.outputs_native.ion_density_fraction)),
    )
    compared = model_comparison(rows, reference="neo")
    return {
        name: compared["models"][name]["integrated_relative_difference"]
        for name in ("sauter", "redl")
    }, compared["effect_size"]["integrated_current"]


if HAVE_SENSITIVITY:
    from vaft.machine_mapping.neoclassical import core_profiles_from_neo

    arms = {
        f"hydrogen + {IMPURITY}": rank_against(result, SOLVER_CHARGE, IMPURITY),
        "hydrogen only": rank_against(
            hydrogen_result,
            float(np.nanmean(hydrogen_result.outputs_native.effective_charge)),
            None,
        ),
    }
    print(f"  {'plasma':20s} {'Z_eff':>6s} {'sauter':>9s} {'redl':>9s} {'I_bs':>9s}   closer")
    for label, (ranking, current) in arms.items():
        charge = SOLVER_CHARGE if IMPURITY in label else 1.0
        closer = min(ranking, key=lambda n: abs(ranking[n]))
        print(f"  {label:20s} {charge:6.2f} {ranking['sauter'] * 100:+8.2f}%"
              f" {ranking['redl'] * 100:+8.2f}% {current / 1e3:8.2f} kA   {closer}")

    both = [r for r, _ in arms.values()]
    between_models = abs(both[0]["sauter"] - both[0]["redl"])
    between_arms = abs(both[0]["redl"] - both[1]["redl"])
    print(f"\n  spread between the models within one plasma : {between_models * 100:.1f}%")
    print(f"  spread in Redl between the two plasmas      : {between_arms * 100:.1f}%")
    print("\nThe assumption moves the answer further than the choice of model does,")
    print("and it reverses which model is closer. Neither ranking is a property of")
    print("the models alone, and neither should be quoted without the plasma.")
elif HAVE_RUN:
    print("Only one arm ran, so the sensitivity cannot be shown. The ranking above")
    print("is for this plasma and this impurity model only.")
else:
    print("Would run the comparison twice -- once with carbon and once without --")
    print("and show that which model looks closer follows the impurity assumption")
    print("rather than the models.")

### And the claim Redl was implemented for

The other thing this comparison was built to test is that the gap between the two
fits widens as the trapped fraction rises -- the 1999 fit was built at
conventional aspect ratio, and that is the argument for preferring the 2021 refit
on a spherical tokamak.

The metric below reports the gap in bins of `f_trap` rather than as a single
number, because on this state the relationship is **not monotonic** and any
single number hides that. Read the bins: the innermost one is large and badly
scattered, because the bootstrap current crosses zero near the axis and a
relative metric there is meaningless however the significance floor is set.
Outside it the gap does rise cleanly to the edge.

So the claim is neither confirmed nor refuted here -- it is contaminated at one
end, and `grows_with_trapping` says `None` with the reason attached rather than
picking a side. That is a change from an earlier version of this notebook, which
compared the mean of the lowest third of `f_trap` with the highest third and
reported `True`; a straight-line fit over the same points reports the opposite
(issue #808).

In [ ]:
# `.get`, not `[...]`: model_comparison omits every metric including the trend when
# two profiles share fewer than two finite radii, which a narrower RMIN_1/RMIN_2 band
# produces. Indexing it would raise there instead of reporting it.
redl_vs_sauter = model_comparison(models, reference="sauter")["models"]["redl"]
trend = redl_vs_sauter.get("trend")
if trend is None:
    print("No trend to report:", redl_vs_sauter.get(
        "reason", "too few radii above the significance floor to bin the gap"
    ))
else:
    print(f"  {'f_trap':>8s} {'Redl vs Sauter':>16s} {'scatter':>9s} {'points':>7s}")
    for entry in trend["bins"]:
        print(f"  {entry['f_trap']:8.2f} {entry['difference'] * 100:15.1f}%"
              f" {entry['stderr'] * 100:8.1f}% {entry['points']:7d}")
    print(f"\n  monotonic in f_trap   : {trend['monotonic']}")
    print(f"  grows with trapping   : {trend['grows_with_trapping']}")
    if "reason" in trend:
        print(f"  why not               : {trend['reason']}")
    print(f"\n  (measured only where the reference carries at least"
          f" {trend['evaluated_above']:.0%} of its peak,")
    print(f"   over {trend['points']} radii)")

## 6. The figure

One series per model on one radial axis. NEO solves a handful of surfaces, so its
line stops where it was not asked -- that gap is an honest statement about the
run, not a rendering artifact, and the plot carries it as a validity mask rather
than interpolating across it.

In [ ]:
fig, ax = vaft.omas.render_plot(
    "neoclassical_profile_bootstrap_current", ods,
    z_eff=Z_EFF, impurity=IMPURITY, rho_range=(0.05, 1.0),
)
ax.set_title(
    f"VEST 48224 at {TIME:.3f} s -- bootstrap current, Z_eff = {Z_EFF} as {IMPURITY}"
)
plt.show()

## 7. What this establishes

**It does establish the size of the effect.** On 48224 the bootstrap current is
between one and two percent of `Ip`, depending on the impurity model -- 1.76 kA
with carbon at `Z_eff = 2`, 2.19 kA for the same discharge treated as pure
hydrogen. This is a cold ohmic discharge and the bootstrap contribution is small;
that is the physics, not a defect in the calculation.

**It establishes that the impurity assumption dominates the model comparison.**
This is the notebook's main result, and it is not the one it was written to find.
Which analytic model sits closer to NEO reverses between the two arms of
section 5: Redl is closer on the pure-hydrogen plasma, Sauter on the carbon one,
and the spread between the two plasmas is larger than the spread between the
models within either. A statement of the form "Redl is the better model for VEST"
cannot be supported by this state without also fixing the impurity content, which
VEST does not measure.

That reverses what an earlier version of this notebook claimed. The cause was
mechanical: `Z_eff` was written into a column of `input.gacode` that NEO does not
read, while the analytic models were evaluated at that same number, so the two
sides described different plasmas (issue #803). Both sides now take the same
impurity model, and `bootstrap_models` refuses a comparison whose charges
disagree.

**It does not establish a measured model error.** This is an *end-to-end*
comparison, not a like-for-like one: the analytic profiles are evaluated from the
ODS's fitted profiles and equilibrium, while NEO ran on the GACODE conversion of
that state. The two sides now share an effective charge and an impurity model,
but not a trapped fraction, a collisionality convention (issue #353) or a radial
grid, so part of any difference remains the input path rather than the model.

The like-for-like check exists and is tight, but it is a *test*, not this
notebook: NEO writes its own Sauter and Redl columns beside its solve, and
`test_formula_neoclassical.py` holds VAFT's kernels against them to 1e-7 in two
different regimes. Read a gap here as a question worth investigating, not as a
number to quote.

**It does not establish anything about the innermost or outermost plasma.**
`psi_N < 0.05` is excluded as untrustworthy, the profiles are truncated at
`rho_max`, and NEO was asked about seven surfaces in between.

**It does not settle whether the gap grows with trapping.** That claim is the
argument for preferring Redl on a spherical tokamak, and on this state it cannot
be tested: the relationship is U-shaped in `f_trap`, contaminated at the inner end
by the bootstrap current's zero crossing and rising cleanly only outside it. The
metric reports the bins and declines the verdict (issue #808) rather than picking
whichever side the summary statistic happens to favour.

**It rests on three assumptions**, all stated in section 0 and none measured on
this shot: the effective charge, the impurity that carries it -- flat in radius,
fully stripped, at the main ion's temperature -- and where to truncate.

## 8. Cleanup

A NEO run is small -- a few hundred kilobytes of `out.neo.*` files -- but it goes
to a temporary directory anyway, because the inputs that produced it are in the
repository and anything here can be regenerated by re-running.

To keep a run for inspection, comment out the `close()` below; the path is
printed in section 3.

In [ ]:
_cleanup.close()
print("done" if RUN_DIR is None else f"removed: {not RUN_DIR.exists()}")